# Do Players Notice the AI, or Just the Quality?
**Author:** Neha Kollikara and Tomas Aguilar

**Course:** INFO 4340 - Analytics and AI

**Question:** Among buyers who dislike AI-disclosed Steam games, how often is generative AI the stated reason, compared with quality? (Q2, exploratory: when reviewers don't mention AI, do complaints concentrate where AI was used?)

**Notebook Parts:**
1. Pilot: keyword search and hand check (development set)
2. Clean: raw scrape to `data/reviews_clean.csv`
3. Keyword baseline *(coming)*
4. Claude classifier *(coming)*
5. Evaluation *(coming)*
6. Analysis *(coming)*

# Setup

In [22]:
#Imports
import pandas as pd
import numpy as np
import os
from scipy import stats

# Part 1: Pilot
**Purpose:** Before the full scrape, estimate how often negative reviews of AI-disclosed games mention generative AI, and test a keyword search.

**Data:** 312 English negative reviews (Jan 2024 – Dec 2025) from the 5 lowest-rated AI-disclosed games at the time, scraped Nov 4, 2026.

**Outputs:** `data/pilot.csv` (development set, excluded from all testing) and `data/pilot_hits_labeled.csv` (hand-checked keyword hits).

**Key results:** 8 of 15 keyword hits (53%) were about generative AI; estimated rate 2.6% (95% CI 1.3–5.0%).

## Pull the Pilot

In [2]:
# Pull the pilot (already run on Nov 4, 2026; left commented out so re-running the notebook doesn't change the data)

# %run steam_reviews_scraper.py --appids 2738870 3484300 2687970 2943670 3444080 --max-per-game 100 --out data/pilot.csv

## Load and Check

In [3]:
# Load the pilot and confirm it matches what was scraped
pilot = pd.read_csv("data/pilot.csv")
print(f"Loaded: {pilot.shape[0]} rows, {pilot.shape[1]} columns")
print(pilot["appid"].value_counts())

assert len(pilot) == 312, f"Expected 312 pilot reviews, found {len(pilot)}"
assert pilot["appid"].nunique() == 5, "Pilot should cover 5 games"
assert (pilot["voted_up"] == False).all(), "Pilot should contain only negative reviews"
assert pilot["recommendationid"].is_unique, "Review IDs should be unique"

Loaded: 312 rows, 18 columns
appid
3484300    100
2687970    100
2738870     48
2943670     41
3444080     23
Name: count, dtype: int64


## Keyword Search

In [4]:
# Keyword search for AI mentions
# This pattern is fixed here and reused unchanged as the keyword baseline, so it isn't tuned on test data
AI_PATTERN = r"\bai\b|a\.i\.|ai-generated|generative|genai|slop|midjourney|chatgpt|artificial intelligence"

hits = pilot[pilot["review"].str.contains(AI_PATTERN, case=False, regex=True)].copy()
print(f"{len(hits)} of {len(pilot)} reviews match the keyword pattern ({len(hits)/len(pilot):.1%})")
print(hits.groupby("appid").size())

15 of 312 reviews match the keyword pattern (4.8%)
appid
2687970     3
2943670     1
3484300    11
dtype: int64


## Export for Hand-Check

In [5]:
# Export keyword hits for hand labeling
# Only writes the file if it doesn't exist, so re-running never erases the hand labels
out_file = "data/pilot_hits_labeled.csv"
if not os.path.exists(out_file):
    hits["label"] = ""
    hits["note"] = ""
    hits[["recommendationid", "appid", "review", "label", "note"]].to_csv(out_file, index=False)
    print(f"Exported {len(hits)} hits to {out_file}")
else:
    print(f"{out_file} already exists; not overwriting hand labels")

data/pilot_hits_labeled.csv already exists; not overwriting hand labels


## Hand-Check Results

In [6]:
# Hand check results
# Codes: G = generative AI, E = enemy/NPC/bot behavior, N = keyword matched but no AI meaning, ? = unclear
labeled = pd.read_csv("data/pilot_hits_labeled.csv")

assert len(labeled) == len(hits), "Every keyword hit should be in the labeled file"
assert labeled["label"].notna().all(), "Some hits are still unlabeled"
assert set(labeled["label"]) <= {"G", "E", "N", "?"}, f"Unexpected labels: {set(labeled['label'])}"

print(labeled["label"].value_counts())

def wilson_ci(k, n, z=1.96):
    """95% Wilson confidence interval for a proportion (works well for small counts)."""
    p = k / n
    center = (p + z**2 / (2*n)) / (1 + z**2 / n)
    half = z * np.sqrt(p*(1-p)/n + z**2 / (4*n**2)) / (1 + z**2 / n)
    return center - half, center + half

n_g = (labeled["label"] == "G").sum()
precision = n_g / len(labeled)
rate = n_g / len(pilot)
low, high = wilson_ci(n_g, len(pilot))
print(f"\nKeyword precision: {n_g} of {len(labeled)} hits ({precision:.0%}) were generative AI")
print(f"Estimated generative-AI rate: {rate:.1%} (95% CI {low:.1%} to {high:.1%})")

label
G    8
E    6
N    1
Name: count, dtype: int64

Keyword precision: 8 of 15 hits (53%) were generative AI
Estimated generative-AI rate: 2.6% (95% CI 1.3% to 5.0%)


## Investigate Edge Cases

In [7]:
# The non-G hits and their notes become examples for the labeling guide
for _, row in labeled[labeled["label"] != "G"].iterrows():
    print(f"[{row['label']}] {row['note'] if pd.notna(row['note']) else ''}\n   {row['review'][:150]}...\n")

[E] 
   This is not a fun game to play in its current state, there is some minor fun to be had with building a house and gear. But the human enemies are all S...

[E] 
   Just a generic survival wanna be game with stupid AI that just runs straight at you
the aiming is over sensitive no matter what settings you use ...fo...

[E] 
   (Update: It's better, but still way too buggy to recommend.)
Guns feels great, but the AI is so broken you'll just get aim botted from bushes before e...

[E] pretty sure it's E but AI mapping? What is that
   Well... let's start with the good, if you have a pretty beefy PC the game is gorgeous(def disable motion blur otherwise its a mess), The voice acting ...

[E] 
   I really wanted to like Dreadzone. It has a gritty, post-apocalyptic vibe concept that could’ve been a hit—but right out of the gate, the pacing felt ...

[E] 
   Let me start off by stating I actually really like the idea of this game a lot. The idea environment is pretty, but the sun rays a

## Pilot Decisions
- Complaints about enemy, NPC, or bot behavior ("the AI just runs at you") count as quality complaints, not generative AI complaints.
- Reviews are labeled by the reviewer's stated reason, so "AI random gen skins" counts as generative AI whether or not AI was actually used.
- "Slop" counts as generative AI only when AI is named or clearly implied.
- Keyword precision was 53%, so a keyword search alone is not reliable; this motivates the Claude classifier.
- These 312 reviews are the development set: the Claude prompt is tuned only on them, and they are removed from all test data.
- Bakery Cafe Simulator was in the pilot but later dropped from the final game list (only 23 negative reviews in the window).

---

# Part 2: Clean
**Purpose:** Clean the raw Steam review scrape, remove the pilot (development) reviews, and draw a random sample of up to 100 negative reviews per game for classification.

**Input:** `data/reviews_raw.csv` (raw scrape, Nov 4, 2026), `data/reviews_raw_replacements.csv` (2 replacement games), `data/games_final.csv` (30 games), `data/pilot.csv` (development set)

**Output:** `data/reviews_clean.csv`

**Summary:** 6,612 raw reviews plus 100 from 2 replacement games --> 2,638 cleaned reviews (1,197 disclosed, 1,441 control) from 30 games. Removed 89 reviews from the 2 replaced games, 200 pilot (dev set) reviews, and 19 non-English reviews, then randomly sampled up to 100 reviews per game (seed 42).

## Load the Data

In [8]:
#Read in the data and confirm
df = pd.read_csv("data/reviews_raw.csv")
games = pd.read_csv("data/games_final.csv")
pilot = pd.read_csv("data/pilot.csv")

print(f"Loaded reviews: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Loaded games: {games.shape[0]} rows")
print(f"Loaded pilot (dev set): {pilot.shape[0]} rows")

Loaded reviews: 6612 rows, 29 columns
Loaded games: 30 rows
Loaded pilot (dev set): 312 rows


## Explore the Data

In [9]:
#Explore the data structure
print("\nINFO:")
df.info()

print("\nDESCRIBE:")
print(df.describe().to_string())

print("\nDUPLICATES:")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate review IDs: {df['recommendationid'].duplicated().sum()}")

print("\nMISSING VALUES:")
print(df.isnull().sum())


INFO:
<class 'pandas.DataFrame'>
RangeIndex: 6612 entries, 0 to 6611
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   appid                          6612 non-null   int64  
 1   recommendationid               6612 non-null   int64  
 2   review                         6612 non-null   str    
 3   voted_up                       6612 non-null   bool   
 4   votes_up                       6612 non-null   int64  
 5   votes_funny                    6612 non-null   int64  
 6   weighted_vote_score            6612 non-null   float64
 7   timestamp_created              6612 non-null   int64  
 8   language                       6612 non-null   str    
 9   steam_purchase                 6612 non-null   bool   
 10  received_for_free              6612 non-null   bool   
 11  written_during_early_access    6612 non-null   bool   
 12  refunded                       6612 non-null   bool 

In [10]:
#Explore the data: content checks specific to this scrape

# Date range: should fall within Jan 1, 2024 – Dec 31, 2025
print("DATE RANGE:")
created = pd.to_datetime(df["timestamp_created"], unit="s")
print(f"Earliest: {created.min()}, Latest: {created.max()}")

# Scrape filters: should be all negative (voted_up False) and all English
print("\nSCRAPE FILTER CHECKS:")
print(df["voted_up"].value_counts(dropna=False))
print(df["language"].value_counts(dropna=False))

# Reviews per game (smallest first) and whether all 30 games returned reviews
print("\nREVIEWS PER GAME:")
per_game = df.groupby(["group", "name"]).size().sort_values()
print(per_game.to_string())
print(f"\nGames with reviews: {df['appid'].nunique()} of {len(games)}")
missing_games = set(games["appid"]) - set(df["appid"])
print(f"Games with no reviews: {missing_games if missing_games else 'none'}")

# Reviews by group (disclosed vs. control)
print("\nREVIEWS BY GROUP:")
print(df["group"].value_counts())

# Pilot reviews in the scrape (dev set; removed later so they aren't used for testing)
print("\nPILOT (DEV SET) OVERLAP:")
in_pilot = df["recommendationid"].isin(pilot["recommendationid"])
print(f"Reviews also in pilot.csv: {in_pilot.sum()}")
print(df[in_pilot].groupby("name").size().to_string())

# Review types that may need handling: empty, very short, checkbox templates, non-English text
print("\nREVIEW TEXT:")
text = df["review"].fillna("").str.strip()
words = text.str.split().str.len()
print(f"Empty reviews: {(text == '').sum()}")
print(f"Very short reviews (1-2 words): {(words.between(1, 2)).sum()}")
print(f"Checkbox-template reviews (contain ☐ or ☑): {text.str.contains('☐|☑').sum()}")
print(f"Mostly non-Latin text (under 50% Latin letters): "
      f"{(text.str.count(r'[A-Za-z]') / text.str.len().clip(lower=1) < 0.5).sum()}")

# Review length in words
print("\nWords per review:")
print(words.describe().round(1).to_string())

DATE RANGE:
Earliest: 2024-01-01 06:47:12, Latest: 2025-12-31 22:54:31

SCRAPE FILTER CHECKS:
voted_up
False    6612
Name: count, dtype: int64
language
english    6612
Name: count, dtype: int64

REVIEWS PER GAME:
group      name                               
disclosed  Night Club Simulator                    41
control    Pharmacy Simulator                      41
disclosed  Cinema Simulator 2025                   45
           Hotel Business Simulator                48
           Shawarma Legend                         56
           Esports Godfather                       56
           FACEMINER                               85
           Serum: Toxic Survival                   85
           Chernobylite 2: Exclusion Zone          87
control    Trash Goblin                           112
disclosed  Lost Rift                              119
control    Tavern Manager Simulator               124
           Ultimate Fishing Simulator 2           128
           Motel Manager Simulator    

## Exploration Observations
- 6,612 reviews from the 30 originally selected games, 29 columns. The two replacement games (Step 1) were scraped separately, so they appear under "Games with no reviews" here.
- The scrape filters worked! Every review is negative, tagged English, and dated Jan 1, 2024 – Dec 31, 2025.
- Disclosed games have about half as many reviews (2,250 vs. 4,362), consistent with the review gap found in prior research. Four control games hit the 500-review cap. Sampling up to 100 per game keeps large games from dominating.
- Counts can differ by one from the earlier summary run (e.g. Motel Manager Simulator: 131 vs. 132) because Steam data changes over time. All analysis uses this dated snapshot.
- 289 reviews overlap with the pilot (dev set): all of DREADZONE (100 of 187), TRIBES 3 (100 of 464), Hotel Business Simulator (48 of 48), and Night Club Simulator (41 of 41). Removing them leaves Hotel Business and Night Club with no test reviews.
    - Decision: both are replaced with two other low-rated AI-disclosed games (Step 1), so the 312 pilot reviews stay a clean dev set.
- 277 reviews (4.2%) are only 1–2 words, and 8 are checkbox templates. These are kept: they are real negative reviews, labeled other if they give no reason.
- 53 reviews (0.8%) are mostly non-Latin characters. These could be other languages tagged as English, or ASCII art; they are inspected before deciding whether to remove them.
- Reviews have a median of 48 words (mean 100, max 1,541). The long tail matters for the cost of the Claude step.

---

## Cleaning Steps

In [11]:
df_clean = df.copy()
print("Starting rows:", len(df_clean))

Starting rows: 6612


## Step 1: Replace Games With No Test Reviews

In [12]:
# 1. Replace games whose reviews are all in the pilot (dev set)
n_before = len(df_clean)
print("Step 1 (replace games) - Starting rows:", n_before)

# Replacement scrape (already run on Nov 4, 2026):
# %run steam_reviews_scraper.py --appid-file data/game_selection/games_replacements.csv --max-per-game 500 --out data/reviews_raw_replacements.csv

dropped_ids = [2738870, 2943670]  # Hotel Business Simulator, Night Club Simulator
replacements = pd.read_csv("data/reviews_raw_replacements.csv")
expected_drop = df_clean["appid"].isin(dropped_ids).sum()

df_clean = df_clean[~df_clean["appid"].isin(dropped_ids)]
df_clean = pd.concat([df_clean, replacements], ignore_index=True)
n_after = len(df_clean)

# Assert the swap: old games gone, new reviews added, review set matches the 30 final games
print("Step 1 (replace games) - Ending rows:", n_after,
      f"(dropped {expected_drop}, added {len(replacements)})")
assert n_after == n_before - expected_drop + len(replacements)
assert not df_clean["appid"].isin(dropped_ids).any(), "Dropped games should be gone"
assert set(df_clean["appid"]) == set(games["appid"]), "Reviews should cover exactly the 30 final games"
assert set(replacements.columns) == set(df.columns), "Replacement scrape should have the same columns"
assert df_clean["recommendationid"].is_unique, "Review IDs should be unique"

# Why: All in-window reviews for Hotel Business Simulator and Night Club Simulator were in the pilot,
# which is the dev set and can't be used for testing. Replacing them with two low-rated AI-disclosed
# management sims (REAL ESTATE Simulator, The Last FARMER) keeps 5 games in each rating tier and the same genre mix.

Step 1 (replace games) - Starting rows: 6612
Step 1 (replace games) - Ending rows: 6623 (dropped 89, added 100)


## Step 2: Parse Review Dates

In [13]:
# 2. Parse review dates
n_before = len(df_clean)
print("Step 2 (dates) - Starting rows:", n_before)

df_clean["created"] = pd.to_datetime(df_clean["timestamp_created"], unit="s")

n_after = len(df_clean)
print("Step 2 (dates) - Ending rows:", n_after)
print(f"Date range: {df_clean['created'].min()} to {df_clean['created'].max()}")

# Assert row count unchanged, every date parsed, and all dates inside the study window
assert n_after == n_before, "Parsing dates shouldn't change row count"
assert df_clean["created"].notna().all(), "Some dates failed to parse"
assert df_clean["created"].between("2024-01-01", "2025-12-31 23:59:59").all(), "Reviews outside the study window"

# Why: created was read from the CSV as text, so it can't be sorted or grouped by month. Parsing it from the
# Unix timestamp gives a true datetime and confirms Steam's date filter also held for the replacement scrape.

Step 2 (dates) - Starting rows: 6623
Step 2 (dates) - Ending rows: 6623
Date range: 2024-01-01 06:47:12 to 2025-12-31 22:54:31


## Step 3: Validate Scrape Filters

In [14]:
# 3. Validate scrape filters
n_before = len(df_clean)
print("Step 3 (filters) - Starting rows:", n_before)
print(df_clean["voted_up"].value_counts(dropna=False))
print(df_clean["language"].value_counts(dropna=False))
n_after = len(df_clean)
print("Step 3 (filters) - Ending rows:", n_after)

# Assert every review is negative and English, including the replacement scrape
assert n_after == n_before
assert (df_clean["voted_up"] == False).all(), "Found a positive review"
assert (df_clean["language"] == "english").all(), "Found a non-English review"

# Why: The study only covers English negative reviews. If Steam's API ever returned other reviews, they would
# silently mix into the sample, so this confirms the filters held for both scrapes.

Step 3 (filters) - Starting rows: 6623
voted_up
False    6623
Name: count, dtype: int64
language
english    6623
Name: count, dtype: int64
Step 3 (filters) - Ending rows: 6623


## Step 4: Remove Pilot (Dev Set) Reviews

In [15]:
# 4. Remove pilot (dev set) reviews
n_before = len(df_clean)
print("Step 4 (pilot) - Starting rows:", n_before)

in_pilot = df_clean["recommendationid"].isin(pilot["recommendationid"])
expected_drop = in_pilot.sum()
print(df_clean[in_pilot].groupby("name").size().to_string())
df_clean = df_clean[~in_pilot].reset_index(drop=True)

n_after = len(df_clean)
print("Step 4 (pilot) - Ending rows:", n_after, f"(dropped {n_before - n_after})")

# Assert the expected rows were removed and no pilot review remains
assert n_after == n_before - expected_drop
assert not df_clean["recommendationid"].isin(pilot["recommendationid"]).any(), "Pilot reviews remain"

# Why: The pilot reviews were used to design the keyword search and will be used to tune the Claude prompt.
# Testing on them would overstate how well both methods work, so they are kept out of all test data.

Step 4 (pilot) - Starting rows: 6623
name
DREADZONE           100
TRIBES 3: Rivals    100
Step 4 (pilot) - Ending rows: 6423 (dropped 200)


## Inspect Possibly Non-English Reviews

In [16]:
# Look at reviews with under 50% Latin letters before deciding whether to remove them
text = df_clean["review"].str.strip()
latin_share = text.str.count(r"[A-Za-z]") / text.str.len().clip(lower=1)
non_latin = df_clean[latin_share < 0.5]

print(f"Mostly non-Latin reviews: {len(non_latin)}")
print(non_latin.groupby(["group", "name"]).size().to_string())
print()
for review in non_latin["review"].head(25):
    print("-", review[:120].replace("\n", " "))

Mostly non-Latin reviews: 51
group      name                               
control    Aloft                                   1
           Drug Dealer Simulator 2                10
           FBC: Firebreak                          4
           Farming Simulator 25                    3
           Kebab Chefs! - Restaurant Simulator     2
           Killing Floor 3                         5
           Sengoku Dynasty                         4
           Sniper Elite: Resistance                3
           Tactical Breach Wizards                 2
           Tavern Manager Simulator                2
           Trash Goblin                            1
           Ultimate Fishing Simulator 2            3
disclosed  DREADZONE                               2
           Esports Godfather                       2
           FACEMINER                               1
           Serum: Toxic Survival                   1
           Shawarma Legend                         3
           TRIBES 3: Ri

## Inspection Observations
- 51 reviews have under 50% Latin letters (2 fewer than in the raw scrape, since those were removed with the pilot or replaced games). They are three different kinds:
    - Written in another language but tagged English: Russian, Ukrainian, Korean, Thai, Arabic (e.g. "не работают сервера! возврат").
    - No words at all: punctuation, emoji, or ASCII art (".", ":(", "💩", "-1").
    - English with many symbols, caught by mistake ("?????? what is the problem of this game").
- The 50% rule mixes these up, so it isn't used to drop reviews. Step 5 drops only reviews where most letters are from another alphabet, which targets non-English reviews.
- Reviews with no letters are kept, consistent with keeping very short reviews: they are real negative reviews and are labeled other.
- Most flagged reviews come from control games (e.g. Drug Dealer Simulator 2: 10), so Step 5 removes slightly more control reviews; the effect is small.

## Step 5: Remove Non-English Reviews

In [17]:
# 5. Remove non-English reviews (written mainly in another alphabet)
n_before = len(df_clean)
print("Step 5 (language) - Starting rows:", n_before)

text = df_clean["review"].fillna("")
letters = text.apply(lambda s: sum(ch.isalpha() for ch in s))   # letters in any alphabet; symbols, emoji, ASCII art don't count
latin_letters = text.str.count(r"[A-Za-zÀ-ɏ]")                  # Latin letters, including accented ones
other_script = (letters > 0) & ((letters - latin_letters) / letters.clip(lower=1) > 0.5)

expected_drop = other_script.sum()
print(f"Non-English reviews: {expected_drop}")
print(df_clean[other_script].groupby("group").size().to_string())
for review in df_clean.loc[other_script, "review"].head(10):
    print("-", review[:80].replace("\n", " "))

df_clean = df_clean[~other_script].reset_index(drop=True)
n_after = len(df_clean)
print("Step 5 (language) - Ending rows:", n_after, f"(dropped {n_before - n_after})")
print(f"Kept reviews with no letters (punctuation, emoji, ASCII art): {(letters[~other_script] == 0).sum()}")

# Assert the expected rows were removed, and only from the 51 flagged in the inspection
assert n_after == n_before - expected_drop
assert expected_drop <= 51, "Should drop no more than the reviews flagged in the inspection"

# Why: The labeling guide, hand labels, and keyword baseline are English only, so reviews written in another language
# can't be labeled consistently. Steam tags them English because that's the reviewer's setting, not the text's language.
# Counting letters only (not symbols) keeps English reviews with lots of punctuation or emoji.
# Non-English reviews written in the Latin alphabet (e.g. Spanish) aren't caught by this rule.

Step 5 (language) - Starting rows: 6423
Non-English reviews: 19
group
control      15
disclosed     4
- у всего один эффект, не интересно, шедуле лучше
 [*]лечето классов персонажи как в героических шутанах
- English. Not many new features except added dodge. Many more flaws. Alpha versio
- Мало контента и долгая прокачка.  Хочется пробовать новые скилы в прокачке, но к
- 병맛좀 곁들인 미완성적 b급 낚시게임
- не работают сервера! возврат
- Один вопрос: "Почему это существует, и кто вообще посчитал разумным пропустить э
- ตอนแรกก็บัคเยอะอยู่แล้ว แทนที่อัพเดตใหม่จะดีขึ้น กากเหมือนเดิม ไม่เข้าใจ เกมก็เส
- بتضل تعلق
 Контента как такого-го мало, только если за
Step 5 (language) - Ending rows: 6404 (dropped 19)
Kept reviews with no letters (punctuation, emoji, ASCII art): 19


**Step 5 Results:** 19 reviews dropped (15 control, 4 disclosed) and 19 no-letter reviews kept. One bilingual review (English summary followed by Ukrainian) was dropped because most of its letters are Ukrainian; its English part was a short summary, so little information is lost.

## Step 6: Add Computed Columns

In [18]:
# 6. Add computed columns
n_before = len(df_clean)
print("Step 6 (computed columns) - Starting rows:", n_before)

df_clean["word_count"] = df_clean["review"].str.split().str.len()
df_clean["playtime_at_review_hrs"] = (df_clean["author_playtime_at_review_min"] / 60).round(1)
df_clean["playtime_forever_hrs"] = (df_clean["author_playtime_forever_min"] / 60).round(1)

n_after = len(df_clean)
print("Step 6 (computed columns) - Ending rows:", n_after)
print(df_clean[["word_count", "playtime_at_review_hrs"]].describe().round(1).to_string())

# Assert row count unchanged and the new columns are complete and valid
assert n_after == n_before, "Adding columns should not change row count"
assert df_clean["word_count"].notna().all() and (df_clean["word_count"] >= 1).all()
assert (df_clean["playtime_at_review_hrs"] >= 0).all()

# Why: Word count is needed to estimate Claude's cost and to check whether very short reviews are labeled differently.
# Playtime in hours is easier to read than minutes and shows whether reviewers played much before reviewing.

Step 6 (computed columns) - Starting rows: 6404
Step 6 (computed columns) - Ending rows: 6404
       word_count  playtime_at_review_hrs
count      6404.0                  6404.0
mean        101.0                    29.1
std         159.7                   122.3
min           1.0                     0.1
25%          18.0                     1.5
50%          49.0                     5.3
75%         117.0                    19.4
max        1541.0                  4254.7


## Step 7: Sample Up to 100 Reviews per Game

In [19]:
# 7. Randomly sample up to 100 reviews per game
n_before = len(df_clean)
print("Step 7 (sample) - Starting rows:", n_before)

per_game_before = df_clean.groupby("appid").size()
expected = per_game_before.clip(upper=100)

# Shuffle with a fixed seed, then keep the first 100 of each game
df_clean = (df_clean.sample(frac=1, random_state=42)
            .groupby("appid").head(100)
            .sort_values(["group", "name", "created"])
            .reset_index(drop=True))

n_after = len(df_clean)
per_game_after = df_clean.groupby("appid").size()
print("Step 7 (sample) - Ending rows:", n_after, f"(dropped {n_before - n_after})")
print(df_clean.groupby("group").size().to_string())

# Assert every game has at most 100, games under 100 kept all their reviews, and all 30 games remain
assert (per_game_after <= 100).all(), "A game has more than 100 reviews"
assert per_game_after.equals(expected.loc[per_game_after.index]), "Games under 100 should keep every review"
assert n_after == expected.sum()
assert df_clean["appid"].nunique() == 30, "All 30 games should remain"

# Why: Some games have hundreds of negative reviews and others under 50. Without capping, a few large control games
# would dominate the results. The random sample (seed 42) is reproducible and avoids the newest-first order Steam returns.

Step 7 (sample) - Starting rows: 6404
Step 7 (sample) - Ending rows: 2638 (dropped 3766)
group
control      1441
disclosed    1197


## Step 8: Save the Cleaned Data

In [20]:
# 8. Save the cleaned data and confirm it reloads correctly
df_clean.to_csv("data/reviews_clean.csv", index=False)
check = pd.read_csv("data/reviews_clean.csv")
print(f"Saved data/reviews_clean.csv: {check.shape[0]} rows, {check.shape[1]} columns")

assert len(check) == len(df_clean), "Saved file should have the same rows"
assert check["recommendationid"].is_unique, "Review IDs should be unique in the saved file"

Saved data/reviews_clean.csv: 2638 rows, 32 columns


## Balance Table

In [21]:
# Compare the disclosed and control groups at the game level
games["is_sim"] = games["genre"].str.contains("sim")
balance = games.groupby("group").agg(
    games=("appid", "count"),
    sim_games=("is_sim", "sum"),
    median_price=("base_price_usd", "median"),
    median_pct_positive=("pct_positive_alltime", "median"),
    median_english_reviews=("total_reviews_alltime", "median"),
    earliest_release=("release", "min"),
    latest_release=("release", "max"),
)
balance["sampled_reviews"] = df_clean.groupby("group").size()
print(balance.T.to_string())
print()
print(pd.crosstab(games["group"], games["rating_tier"]))

group                   control disclosed
games                        15        15
sim_games                     8         9
median_price              19.99     19.99
median_pct_positive        79.7      73.0
median_english_reviews   2324.0     761.0
earliest_release        2024-01   2024-03
latest_release          2025-07   2025-09
sampled_reviews            1441      1197

rating_tier  high  low  mid
group                      
control         6    4    5
disclosed       5    5    5


## Cleaning Observations
- **Final dataset:** 2,638 reviews from 30 games (1,197 disclosed, 1,441 control). The disclosed group is smaller because 9 of its games had fewer than 100 reviews, while 14 of 15 control games reached the cap.
- **Many reviewers played briefly:** a quarter played under 1.5 hours before reviewing (median 5.3 hours), inside Steam's 2-hour refund window. Playtime is very skewed (0.1 to 4,255 hours), so medians are used rather than means.
- **The groups are well matched:** on number of games, genre mix (9 vs. 8 sims), median price ($19.99 for both), release window, and rating tiers (5/5/5 vs. 6/5/4).
- **Two differences are expected and not matched on purpose:** disclosed games are rated lower (73.0% vs. 79.7% positive) and have about a third as many English reviews (761 vs. 2,324). Both may be part of the AI penalty, so matching on them would hide the effect being studied. Because lower-rated games may draw more quality complaints regardless of AI, results are also compared within rating tiers.

## Minimum Detectable Difference

In [23]:
# Smallest difference in per-game complaint rates (disclosed vs. control) detectable with 15 vs. 15 games
# Two-sided test, alpha = 0.05, 80% power. The true game-to-game spread is unknown until labeling, so several are shown.
n_per_group = 15
df_t = 2 * n_per_group - 2
multiplier = stats.t.ppf(0.975, df_t) + stats.t.ppf(0.80, df_t)

mdd = pd.DataFrame({"sd_between_games": [0.05, 0.10, 0.15, 0.20]})
mdd["min_detectable_diff"] = (multiplier * mdd["sd_between_games"] * np.sqrt(2 / n_per_group)).round(3)
print(f"Multiplier: {multiplier:.2f}  (MDD ≈ {multiplier * np.sqrt(2 / n_per_group):.2f} × SD between games)")
print(mdd.to_string(index=False))

Multiplier: 2.90  (MDD ≈ 1.06 × SD between games)
 sd_between_games  min_detectable_diff
             0.05                0.053
             0.10                0.106
             0.15                0.159
             0.20                0.212


## Minimum Detectable Difference Observations
- With 15 games per group, the smallest difference in per-game complaint rates we can reliably detect is about 1.06 × the spread between games (two-sided test, α = 0.05, 80% power).
- If quality complaint rates vary by about 10 points from game to game, only a gap of about 11 points between disclosed and control games would be detectable. Smaller real differences could be missed.
- Generative-AI complaints are rare (about 2.6% in the pilot), so their game-to-game spread is likely small too, which makes a gap of a few points detectable. Control games should be near zero.
- The real spread is unknown until labeling, so this is recomputed with observed values in Part 5. A non-significant result is reported as "no detectable difference," not "no difference."

---

# Part 3: Keyword Baseline

**Purpose:** Flag reviews that match the keyword pattern locked in the pilot. This is the simple, non-AI method the Claude classifier is compared against.

**Input:** `data/reviews_clean.csv` 

**Output:** `data/keyword_flags.csv`

## Load the Cleaned Data

In [24]:
# Load the cleaned reviews
reviews = pd.read_csv("data/reviews_clean.csv")
print(f"Loaded: {reviews.shape[0]} rows, {reviews.shape[1]} columns")

assert len(reviews) == 2638, f"Expected 2,638 cleaned reviews, found {len(reviews)}"
assert reviews["recommendationid"].is_unique

Loaded: 2638 rows, 32 columns


## Apply the Keyword Pattern

In [25]:
# Flag reviews matching the locked pilot pattern (AI_PATTERN from Part 1, unchanged)
reviews["keyword_flag"] = reviews["review"].str.contains(AI_PATTERN, case=False, regex=True)

flags = reviews.groupby("group")["keyword_flag"].agg(flagged="sum", reviews="count")
flags["rate"] = (flags["flagged"] / flags["reviews"]).round(3)
print(flags.to_string())
print(f"\nTotal flagged: {reviews['keyword_flag'].sum()} of {len(reviews)}")

# Save the flags for building the labeling sheet and for evaluation
reviews[["recommendationid", "keyword_flag"]].to_csv("data/keyword_flags.csv", index=False)
assert pd.read_csv("data/keyword_flags.csv").shape[0] == len(reviews)

           flagged  reviews   rate
group                             
control         67     1441  0.046
disclosed      169     1197  0.141

Total flagged: 236 of 2638


## Keyword Baseline Observations
- **236 of 2,638 reviews (8.9%) match the keyword pattern:** 169 of 1,197 disclosed (14.1%) and 67 of 1,441 control (4.6%).
- The disclosed rate is much higher than the pilot's 4.8%, which came from only 5 games. These are keyword matches, not confirmed generative-AI complaints: in the pilot only 53% of matches were.
- Control games also match, likely mostly mentions of enemy or NPC behavior in shooters and survival games.
- Flagged reviews are not printed, so they can be labeled blind in round 1.